In [1]:
import pandas as pd

# rutas (ajusta según tu carpeta)
patients = pd.read_csv("./datasets/PATIENTS.csv")
admissions = pd.read_csv("./datasets/ADMISSIONS.csv")
icustays = pd.read_csv("./datasets/ICUSTAYS.csv")

In [2]:
# convertir fechas
patients["dob"] = pd.to_datetime(patients["dob"])
patients["dod"] = pd.to_datetime(patients["dod"])

admissions["admittime"] = pd.to_datetime(admissions["admittime"])
admissions["dischtime"] = pd.to_datetime(admissions["dischtime"])
admissions["deathtime"] = pd.to_datetime(admissions["deathtime"])

icustays["intime"] = pd.to_datetime(icustays["intime"])
icustays["outtime"] = pd.to_datetime(icustays["outtime"])

In [3]:
df = icustays.merge(admissions, on=["subject_id", "hadm_id"], how="left")
df = df.merge(patients, on="subject_id", how="left")

In [4]:
df["age"] = (df["intime"] - df["dob"]).dt.days / 365

# corregir edades irreales
df.loc[df["age"] > 100, "age"] = 90

In [5]:
df["icu_los_days"] = (df["outtime"] - df["intime"]).dt.total_seconds() / (60*60*24)

In [8]:
df["mortality"] = df["deathtime"].notnull().astype(int)

In [7]:
print(df.columns)

Index(['row_id_x', 'subject_id', 'hadm_id', 'icustay_id', 'dbsource',
       'first_careunit', 'last_careunit', 'first_wardid', 'last_wardid',
       'intime', 'outtime', 'los', 'row_id_y', 'admittime', 'dischtime',
       'deathtime', 'admission_type', 'admission_location',
       'discharge_location', 'insurance', 'language', 'religion',
       'marital_status', 'ethnicity', 'edregtime', 'edouttime', 'diagnosis',
       'hospital_expire_flag', 'has_chartevents_data', 'row_id', 'gender',
       'dob', 'dod', 'dod_hosp', 'dod_ssn', 'expire_flag', 'age',
       'icu_los_days'],
      dtype='str')


In [9]:
dataset = df[[
    "subject_id",
    "hadm_id",
    "icustay_id",
    "gender",
    "age",
    "icu_los_days",
    "mortality"
]]

In [10]:
dataset = dataset.dropna()
dataset = dataset[dataset["icu_los_days"] > 0]

In [1]:
print(dataset.head())
print(dataset.describe())
print(dataset.value_counts("mortality"))

NameError: name 'dataset' is not defined

In [12]:
dataset.to_csv("mimic_base_dataset.csv", index=False)